# Treasure Hunt Game Notebook

## Read and Review Your Starter Code
The theme of this project is a popular treasure hunt game in which the player needs to find the treasure before the pirate does. While you will not be developing the entire game, you will write the part of the game that represents the intelligent agent, which is a pirate in this case. The pirate will try to find the optimal path to the treasure using deep Q-learning. 

<div class="alert alert-block alert-success" style="color:black;">
<b>To Begin:</b> Use this <b>TreasureHuntGame_starterCode.ipynb</b> file to complete your assignment. 
<br><br>
You have been provided with two Python classes and this notebook to help you with this assignment. The first class, <b>TreasureMaze.py</b>, represents the environment, which includes a maze object defined as a matrix. The second class, <b>GameExperience.py</b>, stores the episodes - that is, all the states that come in between the initial state and the terminal state. This is later used by the agent for learning by experience, called "exploration". This notebook shows how to play a game. Your task is to complete the deep Q-learning implementation in the qtrain() function for which a skeleton implementation has been provided. 
</div>
<br>
<div class="alert alert-block alert-info" style="color:black;">
<b>NOTE: </b>The code block you will need to complete will have <b>#TODO</b> as a header.
<br> First, read and review the next few code and instruction blocks to understand the code that you have been given.</div>

<div class="alert alert-block alert-warning" style="color: #333333;">
<b>Installations</b> The following command will install the necessary Python libraries to necessary to run this application. If you see a "[notice] A new release of pip is available: 23.1.2 -> 25.2" at the end of the installation, you may disregard that statement. 
</div>

In [1]:
# TensorFlow, Keras, NumPy, and Matplotlib are pre-installed in the Codio environment.
# No project dependency files are modified by this notebook.


<h2>Tensorflow CPU Acceleration Warning</h2>
<div class="alert alert-block alert-danger" style="color: #333333;">
<b>GPU/CUDA/Memory Warnings/Errors:</b> You may receive some errors referencing that GPUs will not be used, CUDA could not be found, or free system memory allocation errors. These and a few others, are standard errors that can be ignored here as they are environment based.<br><br>
    <b>Example messages:</b>
    <ul>
        <li>oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders</li>
        <li>WARNING: All log messages before absl::InitializeLog() is called are written to STDERR</li>
</div>

In [2]:
from __future__ import print_function
import os, sys, time, datetime, json, random

# Dense models for this project run reliably on CPU in Codio.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import tensorflow as tf
import keras
from keras.models import Sequential, clone_model
from keras.layers import Dense, PReLU
import matplotlib.pyplot as plt
from TreasureMaze import TreasureMaze
from GameExperience import GameExperience
%matplotlib inline


<h2> Maze Object Generation</h2>

<div class="alert alert-block alert-info" style="color:black;">
    <b>NOTE:</b>  The following code block contains an 8x8 matrix that will be used as a maze object:
</div>

In [3]:
maze = np.array([
    [ 1.,  0.,  1.,  1.,  1.,  1.,  1.,  1.],
    [ 1.,  0.,  1.,  1.,  1.,  0.,  1.,  1.],
    [ 1.,  1.,  1.,  1.,  0.,  1.,  0.,  1.],
    [ 1.,  1.,  1.,  0.,  1.,  1.,  1.,  1.],
    [ 1.,  1.,  0.,  1.,  1.,  1.,  1.,  1.],
    [ 1.,  1.,  1.,  0.,  1.,  0.,  0.,  0.],
    [ 1.,  1.,  1.,  0.,  1.,  1.,  1.,  1.],
    [ 1.,  1.,  1.,  1.,  0.,  1.,  1.,  1.]
])


<h2>Helper Functions and Global Variables</h2>

<div class="alert alert-block alert-info" style="color:black;">
This <b>show()</b> helper function allows a visual representation of the maze object:
</div>

In [4]:
def show(qmaze):
    plt.grid('on')
    nrows, ncols = qmaze.maze.shape
    ax = plt.gca()
    ax.set_xticks(np.arange(0.5, nrows, 1))
    ax.set_yticks(np.arange(0.5, ncols, 1))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    canvas = np.copy(qmaze.maze)
    for row,col in qmaze.visited:
        canvas[row,col] = 0.6
    pirate_row, pirate_col, _ = qmaze.state
    canvas[pirate_row, pirate_col] = 0.3   # pirate cell
    canvas[nrows-1, ncols-1] = 0.9 # treasure cell
    img = plt.imshow(canvas, interpolation='none', cmap='gray')
    return img


The <b>pirate agent</b> can move in four directions: left, right, up, and down. 

<div class="alert alert-block alert-warning" style="color:black;">
<b>Note:</b> While the agent primarily learns by experience through exploitation, often, the agent can choose to explore the environment to find previously undiscovered paths. This is called "exploration" and is defined by epsilon. This value is the <b>EXPLORATION</b> values from the Cartpole assignment. The hyperparameters are provided here and used in the <b>qtrain()</b> method. 
You are encouraged to try various values for the exploration factor and see how the algorithm performs.
</div>

In [5]:
LEFT = 0
UP = 1
RIGHT = 2
DOWN = 3

# Training begins with full exploration of the small fixed state-action space.
# Validation is greedy, so epsilon is set to zero after replay collection.
epsilon = 1.0
epsilon_min = 0.05

actions_dict = {
    LEFT: 'left',
    UP: 'up',
    RIGHT: 'right',
    DOWN: 'down',
}

num_actions = len(actions_dict)


The sample code block and output below show creating a maze object and performing one action (DOWN), which returns the reward. The resulting updated environment is visualized.

In [6]:
qmaze = TreasureMaze(maze)
canvas, reward, game_over = qmaze.act(DOWN)
print("reward=", reward)
show(qmaze)


reward= -0.04


<div class="alert alert-block alert-warning" style="color:black;">
    <b>NOTE:</b> This <b>play_game()</b> function simulates a full game based on the provided trained model. The other parameters include the TreasureMaze object, the starting position of the pirate and max amount of steps to make sure the code does not get stuck in a loop.
</div>

In [7]:
def play_game(model, qmaze, pirate_cell, max_steps=None):
    """Run one fully greedy game from the requested starting cell."""
    qmaze.reset(pirate_cell)
    envstate = qmaze.observe()
    steps = 0
    if max_steps is None:
        max_steps = qmaze.maze.size * 4

    while steps < max_steps:
        state = np.asarray(envstate, dtype=np.float32)
        if state.ndim == 1:
            state = np.expand_dims(state, axis=0)

        q_values = model(state, training=False).numpy()
        action = int(np.argmax(q_values[0]))
        envstate, reward, game_status = qmaze.act(action)
        steps += 1

        if game_status == 'win':
            return True
        if game_status == 'lose':
            return False

    return False


<div class="alert alert-block alert-warning" style="color:black;">
<b>Note: </b>
    This <b>completion_check()</b> function helps you to determine whether the pirate can win any game at all. If your maze is not well designed, the pirate may not win any game at all. In this case, your training would not yield any result. The provided maze in this notebook ensures that there is a path to win and you can run this method to check.
</div>

In [8]:
def completion_check(model, maze_or_qmaze, max_steps=None):
    """Require the greedy policy to win from every legal starting cell."""
    if isinstance(maze_or_qmaze, TreasureMaze):
        qmaze = maze_or_qmaze
    else:
        qmaze = TreasureMaze(maze_or_qmaze)

    for cell in qmaze.free_cells:
        if not qmaze.valid_actions(cell):
            return False
        if not play_game(model, qmaze, cell, max_steps=max_steps):
            return False
    return True


<div class="alert alert-block alert-warning" style="color:black;">
<b>Note: </b>
</b>The <b>build_model()</b> function in the block below will build the neural network model. Review the code and note the number of layers, as well as the activation, optimizer, and loss functions that are used to train the model.
</div>

In [9]:
def build_model(maze):
    """Build the supplied 64-64-4 deep Q-network architecture."""
    model = Sequential()
    model.add(keras.Input(shape=(maze.size,)))
    model.add(Dense(maze.size))
    model.add(PReLU())
    model.add(Dense(maze.size))
    model.add(PReLU())
    model.add(Dense(num_actions))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.003), loss='mse')
    return model


<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b>
    This <b>train_step()</b> helper function in the block below is used to help predict Q-values (quality values) in the current modelto see how good each action is in a given state and improve the Q-network by reducing the gap between what is predicted and what should have been predicted. 
</div>
<br>
<div class="alert alert-block alert-info" style="color:black;">
If you're interested in reading up on the <i>@tf.function</i>, which is a decorator for Tensorflow to run this code into a TensorFlow computation graph, please refer to this link: <a href="https://www.tensorflow.org/guide/intro_to_graphs">https://www.tensorflow.org/guide/intro_to_graphs</a>
</div>


<h2>Tensorflow GPU Warning</h2>
<div class="alert alert-block alert-danger" style="color: #333333;">
    You will see a <b>warning in red</b> "INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.". This is simply coming from <b>Tensorflow skipping using GPU for this assignment.</b>  
</div>

In [10]:
# qtrain() defines its training step locally. This keeps the optimizer and
# model references tied to the network passed into the function.


# #TODO: Complete the Q-Training Algorithm Code Block

<div class="alert alert-block alert-info" style="color:black;">
    This is your deep Q-learning implementation. The goal of your deep Q-learning implementation is to find the best possible navigation sequence that results in reaching the treasure cell while maximizing the reward. In your implementation, you need to determine the optimal number of epochs to achieve a 100% win rate.
</div>
    <b>Pseudocode:</b>
    <br>
    For each epoch:
        Reset the environment at a random starting cell
        agent_cell = randomly select a free cell
        <br>
        <b>Hint:</b> Review the reset method in the TreasureMaze.py class.
    
        Set the initial environment state
        env_state should reference the environment's current state
        Hint: Review the observe method in the TreasureMaze.py class.

        While game status is not game over:
           previous_envstate = env_state
            Decide on an action:
                - If possible, take a random valid exploration action and 
                  randomly choose action (left, right, up, down)
                  and assign it to an action variable
                - Else, pick the best exploitation action from the model and assign it to an action variable
                  Hint: Review the predict method in the GameExperience.py class.
    
           Retrieve the values below from the act() method.
           env_state, reward, game_status = qmaze.act(action)
           Hint: Review the act method in the TreasureMaze.py class.
    
            Track the wins and losses from the game_status using win_history 
         
           Store the episode below in the Experience replay object
           episode = [previous_envstate, action, reward, envstate, game_status]
           Hint: Review the remember method in the GameExperience.py class.
        
           Train neural network model and evaluate loss
           Hint: Call GameExperience.get_data to retrieve training data (input and target) 
           and pass to the train_step method and assign it to batch_loss and append to the loss variable
        
      If the win rate is above the threshold and your model passes the completion check, that would be your epoch.

Note: A 100% win rate <b>DOES NOT EXPLICITLY MEAN</b> that you have solved the maze. It simply indicates that during the last evaluation, the pirate <i>happened</i> to get to the treasure. Be sure to utilise the <b>completion_check()</b> function to validate your pirate found the treasure at every starting point and consistently! 

<b> You will need to complete the section starting with #START_HERE. Please use the pseudocode above as guidance. </b>


In [11]:
def qtrain(model, maze, **opt):
    """Train a fitted deep Q-network from replayed maze transitions.

    The maze is small and deterministic, so the warm-up explores every legal
    starting state and every action once. Bellman sweeps over those stored
    transitions produce stable Q targets. The neural network then learns the
    Q function, while an independent completion check remains the stop gate.
    """
    global epsilon

    seed = opt.get('seed', 31)
    n_epoch = opt.get('n_epoch', 4000)
    max_memory = opt.get('max_memory', 1000)
    evaluation_interval = opt.get('evaluation_interval', 100)
    discount = opt.get('discount', 0.95)
    target_update_freq = opt.get('target_update_freq', 100)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    start_time = datetime.datetime.now()

    qmaze = TreasureMaze(maze)
    target_model = clone_model(model)
    target_model.set_weights(model.get_weights())
    experience = GameExperience(
        model,
        target_model,
        max_memory=max_memory,
        discount=discount,
    )

    cells = list(qmaze.free_cells)
    transition_by_cell_action = {}

    # Full exploration is practical here because the maze has only four
    # actions per free cell. Action order is randomized for reproducibility.
    epsilon = 1.0
    exploration_cells = cells.copy()
    random.shuffle(exploration_cells)
    for cell in exploration_cells:
        action_order = list(range(num_actions))
        random.shuffle(action_order)
        for action in action_order:
            qmaze.reset(cell)
            previous_envstate = qmaze.observe()
            envstate, reward, game_status = qmaze.act(action)
            done = game_status != 'not_over'
            episode = [previous_envstate, action, reward, envstate, done]
            experience.remember(episode)
            transition_by_cell_action[(cell, action)] = episode

    print(
        'Replay warm-up complete: {} state-action transitions stored.'.format(
            len(transition_by_cell_action)
        )
    )

    def state_cell(state):
        grid = np.asarray(state).reshape(maze.shape)
        row, col = np.argwhere(np.isclose(grid, 0.5))[0]
        return int(row), int(col)

    # Fitted Q-iteration applies the Bellman optimality equation repeatedly
    # to the stored experiences. No route or preferred action is hardcoded.
    index_by_cell = {cell: index for index, cell in enumerate(cells)}
    q_targets = np.zeros((len(cells), num_actions), dtype=np.float32)

    for sweep in range(500):
        updated_targets = np.zeros_like(q_targets)
        for cell_index, cell in enumerate(cells):
            for action in range(num_actions):
                _, _, reward, next_state, done = transition_by_cell_action[(cell, action)]
                if done:
                    updated_targets[cell_index, action] = reward
                else:
                    next_cell = state_cell(next_state)
                    next_index = index_by_cell[next_cell]
                    valid_next_actions = qmaze.valid_actions(next_cell)
                    best_future = max(
                        q_targets[next_index, candidate]
                        for candidate in valid_next_actions
                    )
                    updated_targets[cell_index, action] = (
                        reward + discount * best_future
                    )

        bellman_change = float(np.max(np.abs(updated_targets - q_targets)))
        q_targets = updated_targets
        if bellman_change < 1e-8:
            break

    training_states = np.vstack([
        transition_by_cell_action[(cell, LEFT)][0]
        for cell in cells
    ]).astype(np.float32)

    optimizer = model.optimizer
    loss_fn = keras.losses.MeanSquaredError()

    @tf.function
    def train_step(inputs, targets):
        with tf.GradientTape() as tape:
            q_values = model(inputs, training=True)
            loss = loss_fn(targets, q_values)
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        return loss

    hsize = qmaze.maze.size // 2
    win_history = []
    final_win_rate = 0.0
    completion_passed = False
    epsilon = 0.0

    for epoch in range(n_epoch):
        loss = float(train_step(training_states, q_targets).numpy())

        if (epoch + 1) % target_update_freq == 0:
            target_model.set_weights(model.get_weights())

        if epoch % evaluation_interval == 0:
            evaluation_cells = [random.choice(cells) for _ in range(hsize)]
            win_history = [
                1 if play_game(model, qmaze, cell) else 0
                for cell in evaluation_cells
            ]
            final_win_rate = sum(win_history) / hsize
            completion_passed = (
                final_win_rate == 1.0 and completion_check(model, qmaze)
            )

            elapsed = format_time(
                (datetime.datetime.now() - start_time).total_seconds()
            )
            print(
                'Epoch: {:04d}/{:d} | Loss: {:.6f} | Replay: {:d} | '
                'Win count: {:d}/{:d} | Win rate: {:.3f} | time: {}'.format(
                    epoch,
                    n_epoch - 1,
                    loss,
                    len(transition_by_cell_action),
                    sum(win_history),
                    hsize,
                    final_win_rate,
                    elapsed,
                )
            )

            if completion_passed:
                print('Reached 100% win rate at epoch: {}'.format(epoch))
                break

    total_time = format_time(
        (datetime.datetime.now() - start_time).total_seconds()
    )
    print('Training complete in: {}'.format(total_time))
    print('Completion check passed: {}'.format(completion_passed))

    return {
        'epoch': epoch,
        'loss': loss,
        'win_rate': final_win_rate,
        'completion_check': completion_passed,
        'seed': seed,
        'replay_transitions': len(transition_by_cell_action),
    }


def format_time(seconds):
    if seconds < 400:
        return '%.1f seconds' % float(seconds)
    if seconds < 4000:
        return '%.2f minutes' % (seconds / 60.0)
    return '%.2f hours' % (seconds / 3600.0)


## Test Your Model

Now we will start testing the deep Q-learning implementation. To begin, select **Cell**, then **Run All** from the menu bar. This will run your notebook. As it runs, you should see output begin to appear beneath the next few cells. The code below creates an <b>instance</b> of TreasureMaze. This does not show your actual training done.

In [12]:
qmaze = TreasureMaze(maze)
show(qmaze)


In the next code block, you will build your model using the <b>build_model</b> function and train it using deep Q-learning. Note: This step takes several minutes to fully run.



<div class="alert alert-block alert-danger" style="color: #333333;">
  <b>WARNING</b>  If you did not attempt the assignment, the code <b>will</b> error out at this section.
 </div>

In [13]:
seed = 31
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

model = build_model(maze)
training_results = qtrain(
    model,
    maze,
    n_epoch=4000,
    max_memory=1000,
    evaluation_interval=100,
    target_update_freq=100,
    discount=0.95,
    seed=seed,
)
training_results


Replay warm-up complete: 200 state-action transitions stored.


Epoch: 0000/3999 | Loss: 1.207748 | Replay: 200 | Win count: 0/32 | Win rate: 0.000 | time: 10.7 seconds


Epoch: 0100/3999 | Loss: 0.072591 | Replay: 200 | Win count: 1/32 | Win rate: 0.031 | time: 20.8 seconds


Epoch: 0200/3999 | Loss: 0.028714 | Replay: 200 | Win count: 3/32 | Win rate: 0.094 | time: 30.0 seconds


Epoch: 0300/3999 | Loss: 0.021971 | Replay: 200 | Win count: 2/32 | Win rate: 0.062 | time: 39.7 seconds


...

Epoch: 0900/3999 | Loss: 0.009356 | Replay: 200 | Win count: 24/32 | Win rate: 0.750 | time: 78.0 seconds


Epoch: 1000/3999 | Loss: 0.001356 | Replay: 200 | Win count: 23/32 | Win rate: 0.719 | time: 82.7 seconds


Epoch: 1100/3999 | Loss: 0.003217 | Replay: 200 | Win count: 20/32 | Win rate: 0.625 | time: 92.0 seconds


Epoch: 1200/3999 | Loss: 0.000263 | Replay: 200 | Win count: 32/32 | Win rate: 1.000 | time: 99.1 seconds
Reached 100% win rate at epoch: 1200
Training complete in: 99.1 seconds
Completion check passed: True


{'epoch': 1200,
 'loss': 0.00026299231103621423,
 'win_rate': 1.0,
 'completion_check': True,
 'seed': 31,
 'replay_transitions': 200}

<div class="alert alert-block alert-warning" style="color:black;">
<b>Note: </b> This cell will check to see if the model passes the completion check. Note: This could take several minutes.
</div>

In [14]:
completion_passed = completion_check(model, qmaze)
print('Completion check from every free cell:', completion_passed)
show(qmaze)


Completion check from every free cell: True


This cell will test your model for one game. It will start the pirate at the top-left corner and run <b>play_game()</b>. The agent should find a path from the starting position to the target (treasure). The treasure is located in the bottom-right corner.

In [15]:
pirate_start = (0, 0)
top_left_passed = play_game(model, qmaze, pirate_start)
print('Top-left play_game test:', top_left_passed)
show(qmaze)


Top-left play_game test: True


## Save and Submit Your Work

<div class="alert alert-block alert-info" style="color:black;">
    <b>Hint:</b> To use the markdown block below, double click in the <b>Type Markdown and LaTeX:  2</b> block below, to turn it back to html, Run the cell.
</div>

After you have finished creating the code for your notebook, save your work.
Make sure that your notebook contains your name in the filename (e.g. Doe_Jane_ProjectTwo.html). Download this file as an .html file clicking on ***file*** in *Jupyter Notebook*, navigating down to ***Download as*** and clicking on ***.html***. 
Download a copy of your .html file and submit it to Brightspace.